In [26]:
# --- Autoreload & local import path (run once per kernel) ---
%load_ext autoreload
%autoreload 2

# Ensure repository root is importable when running from examples/
import sys
from pathlib import Path

print("Python executable:", sys.executable)

repo_root = Path.cwd().resolve().parents[0]  # examples/ -> repo root
if (repo_root / "pyproject.toml").exists() is False:
    # Fallback: if CWD isn't examples/, walk upwards
    for parent in Path.cwd().resolve().parents:
        if (parent / "pyproject.toml").exists() or (parent / "setup.py").exists():
            repo_root = parent
            break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root on sys.path:", repo_root)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Python executable: /home/thomas-funck/anaconda3/envs/myenv/bin/python
Repo root on sys.path: /media/windows/projects/neuromaps-prime-model


In [ ]:
# Sanity check: confirm we're importing the local package copy
import nmp_modeling
print("nmp_modeling imported from:", nmp_modeling.__file__)

In [27]:
# import .mat with functional and structural connectivity data
input_file = './Human_66.mat'
import numpy as np
from scipy.io import loadmat
import h5py
mat = loadmat(input_file)

# print the keys of the loaded .mat file to understand its structure
for key, item in mat.items():
    print(f"{key}: {type(item)}")
    # If the item is a numpy array, print its shape
    if isinstance(item, np.ndarray):
        print(f"  Shape: {item.shape}")

n = mat['FC_emp'].shape[0]  # number of nodes
print(f"Number of nodes: {n}")

__header__: <class 'bytes'>
__version__: <class 'str'>
__globals__: <class 'list'>
Order: <class 'numpy.ndarray'>
  Shape: (1, 66)
C: <class 'numpy.ndarray'>
  Shape: (66, 66)
L: <class 'numpy.ndarray'>
  Shape: (66, 66)
FC_emp: <class 'numpy.ndarray'>
  Shape: (66, 66)
anat_lbls: <class 'numpy.ndarray'>
  Shape: (66,)
talairach_66: <class 'numpy.ndarray'>
  Shape: (66, 3)
Number of nodes: 66


In [ ]:


import numpy as np
from nmp_modeling.parametrization import MapParametrization, FreeParam
from nmp_modeling import observables
from nmp_modeling.fitting import grid_sweep
from nmp_modeling.adapters.NeuronumbaAdapter import NeuronumbaAdapter
import numpy as np

gaba_map = np.random.rand(n)  # example brain map, replace with actual data
SC = mat['C']  # structural connectivity matrix
emp_fc = mat['FC_emp']  # empirical functional connectivity matrix

# 1. Describe how a brain map enters the model.
# SKIP this for the Deco 2014 model because there aren't any parameters to fit

# 2. Build a backend-specific adapter.
adapter = NeuronumbaAdapter(
    weights=SC,
    parametrizations=[],
    fixed_model_attrs={"auto_fic": True},
)

# 4. Sweep.
result = grid_sweep(
    adapter=adapter,
    free_grid={"G": np.arange(0, 2.5, 0.25)},
    fixed={},
    observable=observables.fc_fisher_z,
    empirical_target=emp_fc,
    distance=observables.pearson_lower_triangle,
    n_subjects=15,
    run_seeds=[11, 23, 37],
)
print(result.best_theta, result.best_loss)

[grid_sweep] G=0
